# 07 — Multi-turn conversations with tools

When building applications with multiple tools, you need to handle scenarios where Claude might need to call several tools in sequence to answer a single user question. For example, if a user asks "What day is 103 days from today?", Claude needs to first get the current date, then add 103 days to it.

This creates a multi-turn conversation pattern where Claude makes multiple tool requests before providing a final answer. Your application needs to handle this automatically.

### The Multi-Turn Tool Pattern
Here's what happens behind the scenes when Claude needs multiple tools:

1. User asks: "What day is 103 days from today?"
2. Claude responds with a tool use block requesting get_current_datetime
3. Your server calls the function and returns the result
4. Claude realizes it needs more information and requests add_duration_to_datetime
5. Your server calls that function and returns the result
6. Claude now has enough information to provide the final answer

### Pseudocode

```python
def run_conversation(messages):
    while True:
        response = chat(messages)
        
        add_user_message(messages, response)
        
        # Pseudo code
        if response isn't asking for a tool:
            break
            
        tool_result_blocks = run_tools(response)
        add_user_message(tool_result_blocks)
        
    return messages
```

Before implementing this, we need to update the `add_user_message` and `add_assistant_message` helper functions.

In [3]:
# Add repo root to path
import sys
sys.path.append("../..")

#  Imports
from src.utils import get_client, get_model
from src.utils import add_assistant_message, add_user_message, chat
from src.tool_use import get_current_datetime, get_current_datetime_schema


client = get_client()
model = get_model()

In [5]:
messages = []

add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)
add_assistant_message(messages, response)

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current time formatted as HH:MM:SS for you.", type='text'),
   ToolUseBlock(id='toolu_01F2335uTpETEc8JUSzKTgsE', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [6]:
from anthropic.types import Message
isinstance(response, Message)

True

### Key Improvements
These refactoring steps prepare your code for robust tool handling:

- **Flexible message handling** - Your helper functions can now work with different message formats
- **Tool support in chat** - The chat function can receive and pass through tool schemas
- **Full message returns** - You get complete message objects instead of just text, preserving all blocks
- **Text extraction utility** - Easy way to get readable text from complex messages

With these foundations in place, you're ready to implement the conversation loop that handles multiple tool calls automatically, creating a seamless experience where Claude can use as many tools as needed to answer user questions.